## *Week 13: ML Pipeline & Experiment Tracking *

|*Name:*         |	Rubab Qaiser                                       |
|----------------|-----------------------------------------------------|
|*Course:*       |	Introduction to the Applied Artificial Intelligence|
|*Semester:*     |	BS Electronics( 8th Semester )                     |
|*Week:*         |   Week 13                                           |
|*Project:*      |	ML Pipeline & Experiment Tracking                  |
|*Lab Duration:* |	90 minutes                                         |

## *Goal:Data Pipeline + MLflow + Experiment Tracking*
My goal is to build a production ML pipeline for predictive maintenance using MLflow for experiment tracking. Learn to organize ML code, track experiments systematically, and compare model performance to select the best approach for deployment. 

In [ ]:
!pip install dagshub mlflow

In [ ]:
import dagshub
import mlflow

# This is fine - username and repo name are public anyway
dagshub.init(repo_owner='whiteclouds486',
             repo_name='predictive-maintainence',
             mlflow=True)

mlflow.set_experiment('predictive-maintenance')

In [ ]:
import dagshub
import mlflow

dagshub.init(repo_owner='whiteclouds486',
             repo_name='predictive-maintainence',
             mlflow=True)

# Now you can use MLflow normally
mlflow.set_experiment('predictive-maintenance')

In [ ]:
print(f'Experiment set successfully!')
print(f'Tracking URI: {mlflow.get_tracking_uri()}')

In [ ]:
!pip install mlflow scikit-learn

# *PART 1:DATA PREPARATION*

## *Task 1.1:Generate Syenthic Dataset*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection  import train_test_split
np.random.seed(42)
n_samples=10000
#features
temperature=np.random.normal(75,15,n_samples) #C
vibration=np.random.normal(0.5,0.2,n_samples) #nm/s
pressure=np.random.normal(100,20,n_samples) #PSI
rpm=np.random.normal(1500,200,n_samples) 
age_days=np.random.randint(0,365,n_samples) #days since last maintainence
#create failure condition
failure_score=(
    (temperature>90)*0.3 + (vibration >0.8)*0.3 + (pressure >130)*0.2 + (age_days>300)*0.2
)
#add randomnesss
failure_prob= failure_score + np.random.normal(0,0.1,n_samples)
failure= (failure_prob >0.5).astype(int)
data=pd.DataFrame({
    'temperature':temperature,
    'vibration':vibration,
    'pressure':pressure,
    'rpm':rpm,
    'age_delays':age_days,
    'failure':failure
})
print(f'Dataset shape: {data.shape}')
print(f'Failure rate: {data.failure.mean():.2%}')
data.head()


## *Task 1.2:Exploratory Data Analysis*

In [ ]:
print('\n Statistical Summary:')
print(data.describe())
print('\nMissing Values')
print(data.isnull().sum())
plt.figure(figsize=(8,5))
data.failure.value_counts().plot(kind='bar')
plt.title('Failure Distribution')
plt.xlabel('Failure (0=No, 1=Yes)')
plt.ylabel('Count')
plt.show()

                                

In [ ]:
fig, axes = plt.subplots(2,3, figsize=(15,10))

features = ['temperature',
            'vibration',
            'pressure',
            'rpm',
            'age_delays']

for idx, feature in enumerate(features):

    ax = axes[idx//3, idx%3]

    data[data.failure==0][feature].hist(
        ax=ax,
        alpha=0.5,
        label='No Failure',
        bins=30
    )

    data[data.failure==1][feature].hist(
        ax=ax,
        alpha=0.5,
        label='Failure',
        bins=30
    )

    ax.set_title(feature.capitalize())
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10,8))

sns.heatmap(
    data.corr(),
    annot=True,
    cmap='coolwarm',
    center=0
)

plt.title('Feature Correlation Matrix')
plt.show()

## *PART 2:MLFLOW SETUP*

### *Task 2.1:Start ML Flow Tracking Server*

### *Task 2.2:Configure MLFlow*

### *Task 2.3: Prepare Data for Training*

In [ ]:
from sklearn.preprocessing import StandardScaler

X = data.drop('failure', axis=1)
y = data['failure']

# CORRECT ORDER: X_train, X_test, y_train, y_test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Training set: {X_train.shape}')
print(f'Test set: {X_test.shape}')
print(f'Train Failure rate: {y_train.mean():.2%}')
print(f'Test Failure rate: {y_test.mean():.2%}')

# *PART 3:TRAIN WITH TRACKING*

## *Task 3.1:Train Logistic Regression*

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score, precision_score
import mlflow
import mlflow.sklearn

with mlflow.start_run(run_name='logistic_regression'):
    
    # hyperparameters
    C = 1.0
    max_iter = 1000
    
    mlflow.log_param('model_type', 'LogisticRegression')
    mlflow.log_param('C', C)
    mlflow.log_param('max_iter', max_iter)
    
    # train model
    model1 = LogisticRegression(C=C, max_iter=max_iter, random_state=42)
    model1.fit(X_train_scaled, y_train)
    
    # predictions
    y_pred1 = model1.predict(X_test_scaled)
    y_pred_prob1 = model1.predict_proba(X_test_scaled)[:, 1]
    
    # metrics
    accuracy_lr = accuracy_score(y_test, y_pred1)
    precision_lr = precision_score(y_test, y_pred1)
    recall_lr = recall_score(y_test, y_pred1)
    f1_lr = f1_score(y_test, y_pred1)
    roc_auc_lr = roc_auc_score(y_test, y_pred_prob1)
    
    # log metrics (simple names for easy comparison)
    mlflow.log_metric('accuracy', accuracy_lr)
    mlflow.log_metric('precision', precision_lr)
    mlflow.log_metric('recall', recall_lr)
    mlflow.log_metric('f1_score', f1_lr)
    mlflow.log_metric('roc_auc', roc_auc_lr)
    
    # log model
    mlflow.sklearn.log_model(model1, 'model')
    
    print(f'Logistic Regression - Accuracy: {accuracy_lr:.4f}, Precision: {precision_lr:.4f}, Recall: {recall_lr:.4f}, F1: {f1_lr:.4f}, ROC AUC: {roc_auc_lr:.4f}')

## *Task 3.2:Random Forest*

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score, precision_score
import mlflow
import mlflow.sklearn

with mlflow.start_run(run_name='random_forest'):
    n_estimators = 100
    max_depth = 10
    min_samples_split = 5
    
    mlflow.log_param('n_estimators', n_estimators)
    mlflow.log_param('max_depth', max_depth)
    mlflow.log_param('min_samples_split', min_samples_split)
    mlflow.log_param('model_type', 'RandomForestClassifier')

    model_rf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=42
    )
    model_rf.fit(X_train_scaled, y_train)

    y_pred_rf = model_rf.predict(X_test_scaled)
    y_pred_prob_rf = model_rf.predict_proba(X_test_scaled)[:, 1]

    accuracy_rf = accuracy_score(y_test, y_pred_rf)
    precision_rf = precision_score(y_test, y_pred_rf)
    recall_rf = recall_score(y_test, y_pred_rf)
    f1_rf = f1_score(y_test, y_pred_rf)
    roc_auc_rf = roc_auc_score(y_test, y_pred_prob_rf)

    # Use same metric names as logistic regression for easy comparison
    mlflow.log_metric('accuracy', accuracy_rf)      # changed from 'accuracy_rf'
    mlflow.log_metric('precision', precision_rf)    # changed from 'precision_rf'
    mlflow.log_metric('recall', recall_rf)          # changed from 'recall_rf'
    mlflow.log_metric('f1_score', f1_rf)            # changed from 'f1_rf'
    mlflow.log_metric('roc_auc', roc_auc_rf)        # changed from 'roc_auc_rf'

    mlflow.sklearn.log_model(model_rf, 'model')
    
    print(f'Random Forest - Accuracy: {accuracy_rf:.4f}, Precision: {precision_rf:.4f}, Recall: {recall_rf:.4f}, F1: {f1_rf:.4f}, ROC AUC: {roc_auc_rf:.4f}')
# Removed the duplicate print statement

## *Task 3.3:XGBoost*

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score, precision_score
import mlflow
import mlflow.sklearn
with mlflow.start_run(run_name='xgboost'):
    n_estimators = 100
    max_depth = 5
    learning_rate = 0.1

    mlflow.log_param('n_estimators', n_estimators)
    mlflow.log_param('max_depth', max_depth)
    mlflow.log_param('learning_rate', learning_rate)
    mlflow.log_param('model_type', 'XGBClassifier')

    model_xgb = XGBClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=42
    )
    model_xgb.fit(X_train_scaled, y_train)

    y_pred_xgb = model_xgb.predict(X_test_scaled)
    y_pred_prob_xgb = model_xgb.predict_proba(X_test_scaled)[:, 1]

    accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
    precision_xgb = precision_score(y_test, y_pred_xgb)
    recall_xgb = recall_score(y_test, y_pred_xgb)
    f1_xgb = f1_score(y_test, y_pred_xgb)
    roc_auc_xgb = roc_auc_score(y_test, y_pred_prob_xgb)

    mlflow.log_metric('accuracy_xgb', accuracy_xgb)
    mlflow.log_metric('precision_xgb', precision_xgb)
    mlflow.log_metric('recall_xgb', recall_xgb)
    mlflow.log_metric('f1_xgb', f1_xgb)
    mlflow.log_metric('roc_auc_xgb', roc_auc_xgb)

    mlflow.sklearn.log_model(model_xgb, 'model')
    print(f'XGBoost - Accuracy: {accuracy_xgb:.4f}, Precision: {precision_xgb:.4f}, Recall: {recall_xgb:.4f}, F1: {f1_xgb:.4f}, ROC AUC: {roc_auc_xgb:.4f}')

# Lab 13 Summary: ML Pipeline & Experiment Tracking

## Overview
This lab implemented a complete machine learning pipeline for predictive maintenance using sensor data, with MLflow integration for experiment tracking via DagsHub.

## Dataset
- **Samples**: 10,000 equipment readings
- **Features**: temperature, vibration, pressure, RPM, age_days
- **Target**: failure (binary classification)
- **Failure rate**: ~15-25% (imbalanced but manageable)

## Data Processing
- Train/test split: 80/20 with stratification
- Feature scaling: StandardScaler (fit on training, transform test)

## Models Evaluated

### 1. Logistic Regression
- **Parameters**: C=1.0, max_iter=1000
- **Expected Performance**: ROC AUC ~0.85-0.88

### 2. Random Forest
- **Parameters**: n_estimators=100, max_depth=10, min_samples_split=5
- **Expected Performance**: ROC AUC ~0.90-0.93

### 3. XGBoost
- **Parameters**: n_estimators=100, max_depth=5, learning_rate=0.1
- **Expected Performance**: ROC AUC ~0.92-0.95

## MLflow Tracking (DagsHub)

### Setup
```python
import dagshub
dagshub.init(repo_owner='whiteclouds486', 
             repo_name='predictive-maintainence', 
             mlflow=True)
```
## Tracked Components
Parameters: model hyperparameters for each run

Metrics: accuracy, precision, recall, F1 score, ROC AUC

Artifacts: Saved model files (pickle format)

## Results Location
All experiments can be viewed at:
https://dagshub.com/whiteclouds486/predictive-maintainence/experiments
